<a href="https://colab.research.google.com/github/bimaadiwijaya8/-ML_05_Bima-Adiwijaya/blob/main/JS03/JS03-Tugas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

In [4]:
df = pd.read_csv("wbc (1).csv")

df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [5]:
df = df.drop(columns=["id", "Unnamed: 32"], errors="ignore")

X = df.drop(columns=["diagnosis"])
y = df["diagnosis"]

In [6]:
y = y.map({"B": 0, "M": 1})

In [7]:
print(y.value_counts())

diagnosis
0    357
1    212
Name: count, dtype: int64


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [9]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("selector", SelectKBest(score_func=f_classif, k=5)),
    ("model", LogisticRegression(max_iter=1000))
])

In [10]:
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Akurasi:", accuracy)
print(classification_report(y_test, y_pred))

Akurasi: 0.9649122807017544
              precision    recall  f1-score   support

           0       0.96      0.99      0.97        72
           1       0.97      0.93      0.95        42

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114



In [11]:
hasil = []

for k in range(1, 31):
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("selector", SelectKBest(score_func=f_classif, k=k)),
        ("model", LogisticRegression(max_iter=1000))
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    hasil.append({
        "Jumlah Fitur": k,
        "Akurasi": accuracy
    })

hasil_df = pd.DataFrame(hasil)

print(hasil_df)

    Jumlah Fitur   Akurasi
0              1  0.929825
1              2  0.956140
2              3  0.956140
3              4  0.956140
4              5  0.964912
5              6  0.964912
6              7  0.964912
7              8  0.964912
8              9  0.973684
9             10  0.956140
10            11  0.973684
11            12  0.973684
12            13  0.973684
13            14  0.982456
14            15  0.973684
15            16  0.973684
16            17  0.973684
17            18  0.982456
18            19  0.982456
19            20  0.982456
20            21  0.973684
21            22  0.973684
22            23  0.973684
23            24  0.973684
24            25  0.973684
25            26  0.973684
26            27  0.973684
27            28  0.964912
28            29  0.964912
29            30  0.964912


In [12]:
terbaik = hasil_df.loc[hasil_df["Akurasi"].idxmax()]

print("Jumlah fitur terbaik:", terbaik["Jumlah Fitur"])
print("Akurasi terbaik:", terbaik["Akurasi"])

Jumlah fitur terbaik: 14.0
Akurasi terbaik: 0.9824561403508771


In [13]:
k_terbaik = int(terbaik["Jumlah Fitur"])

pipeline_terbaik = Pipeline([
    ("scaler", StandardScaler()),
    ("selector", SelectKBest(score_func=f_classif, k=k_terbaik)),
    ("model", LogisticRegression(max_iter=1000))
])

pipeline_terbaik.fit(X_train, y_train)

fitur_terpilih = X.columns[
    pipeline_terbaik.named_steps["selector"].get_support()
]

print("Jumlah fitur terbaik:", k_terbaik)
print("Fitur yang terpilih:")
print(fitur_terpilih.tolist())

Jumlah fitur terbaik: 14
Fitur yang terpilih:
['radius_mean', 'perimeter_mean', 'area_mean', 'compactness_mean', 'concavity_mean', 'concave points_mean', 'radius_se', 'perimeter_se', 'radius_worst', 'perimeter_worst', 'area_worst', 'compactness_worst', 'concavity_worst', 'concave points_worst']
